# 1️⃣ 缺失值处理（Handling Missing Values）
处理数据集中缺失值是建模前的第一步：
- 删除缺失值
- 使用中位数、平均数或众数填补
- 使用 sklearn 的 SimpleImputer

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# 示例数据
sample_data = pd.DataFrame({
    'Age': [25, np.nan, 30, 22, np.nan],
    'Salary': [50000, 54000, np.nan, 58000, 60000]
})

# 使用中位数填补
imputer = SimpleImputer(strategy='median')
filled_data = pd.DataFrame(imputer.fit_transform(sample_data), columns=sample_data.columns)
filled_data

# 2️⃣ 分类变量处理（Handling Categorical Variables）
分类变量编码策略：
- One-Hot Encoding：适合低基数分类
- Label Encoding：适合高基数或树模型
- 也可以删除无用的分类特征

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# 示例分类数据
df = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green', 'Blue']
})

encoder = OneHotEncoder(sparse=False)
onehot = encoder.fit_transform(df[['Color']])
pd.DataFrame(onehot, columns=encoder.get_feature_names_out(['Color']))

# 3️⃣ 模型验证（Model Validation）
- 使用 `train_test_split` 划分训练/验证集
- 或使用 KFold / cross_val_score 交叉验证
- 常见指标：MAE、RMSE（回归）；Accuracy、F1（分类）

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 示例数据（回归）
X = filled_data[['Age']]
y = filled_data['Salary']

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(X_train, y_train)
preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)
print("MAE:", mae)

# 4️⃣ 使用 Pipeline（流水线）
优点：
- 简化流程
- 防止数据泄露
- 更易部署和调参

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

transformed = pipe.fit_transform(sample_data)
pd.DataFrame(transformed, columns=sample_data.columns)

# 5️⃣ XGBoost 简介与实战
- 高效、可并行的梯度提升树模型
- 内置缺失值处理
- 支持 early stopping

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train,
          early_stopping_rounds=5,
          eval_set=[(X_val, y_val)],
          verbose=False)

xgb_preds = model.predict(X_val)
print("XGBoost MAE:", mean_absolute_error(y_val, xgb_preds))

# 6️⃣ 避免数据泄露（Avoiding Data Leakage）
常见泄露类型：
- **目标泄露**：包含了预测目标的信息（如未来数据）
- **验证集污染**：预处理过程中使用了验证集数据

解决方法：
- 预处理应在训练集上拟合并应用于验证集
- 慎重选择特征，排除与目标高度相关的未来信息